# Notebook 02: Preprocesamiento y particion

Toma el archivo crudo, ejecuta las cinco etapas de limpieza y deja en Drive los conjuntos de
entrenamiento y prueba que consumen los notebooks 03, 04 y 05.

---

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

PROYECTO = "food_delivery_time_prediction"

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = f"/content/drive/MyDrive/{PROYECTO}"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(f"./{PROYECTO}")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)


def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=150, bbox_inches="tight")
    print("Figura guardada:", destino)


print("Persistencia en Drive:", EN_DRIVE)
print("Ruta de trabajo:", RUTA)

In [ ]:
URL = "https://raw.githubusercontent.com/Vikranth3140/Food-Delivery-Time-Prediction/main/datasets/kaggle/train.csv"

df = pd.read_csv(URL)
filas_iniciales = len(df)
print("Filas iniciales:", filas_iniciales)

## 3.2 Preprocesamiento

El diagnostico del notebook 01 identifico cuatro problemas de naturaleza distinta, y cada uno
necesita un tratamiento propio:

1. **Formato de texto.** Espacios sobrantes y prefijos pegados a los valores.
2. **Tipos mal declarados.** Columnas numericas que llegan como texto, y nulos disfrazados
   de la cadena `"NaN"`.
3. **Informacion dispersa.** La distancia no existe como columna: hay que construirla desde
   cuatro columnas de coordenadas.
4. **Registros invalidos.** Coordenadas en cero y calificaciones fuera del rango posible.

El orden importa. La conversion de tipos tiene que ir despues de la limpieza de texto,
porque `" 37 "` con espacios no se convierte a numero. Y el filtrado de invalidos tiene que
ir al final, cuando ya existen todas las columnas sobre las que se filtra.

### Etapa 1: normalizacion de texto

Los valores de texto arrastran espacios al final y dos columnas traen un prefijo constante
pegado al dato. `Time_taken(min)` llega como `"(min) 24"` y `Weatherconditions` como
`"conditions Sunny"`. Son prefijos sin informacion: aparecen en todas las filas por igual, y
mientras esten ahi la columna no se puede convertir a numero ni agrupar por categoria.

In [ ]:
df.columns = [c.strip() for c in df.columns]

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype(str).str.strip()

df["minutos"] = df["Time_taken(min)"].str.replace("(min) ", "", regex=False)
df["clima"] = df["Weatherconditions"].str.replace("conditions ", "", regex=False)

print("Objetivo antes y despues del prefijo:")
print("  crudo:", df["Time_taken(min)"].iloc[0], " limpio:", df["minutos"].iloc[0])
print("  crudo:", df["Weatherconditions"].iloc[0], " limpio:", df["clima"].iloc[0])

### Etapa 2: tipos y nulos disfrazados

Cinco columnas que son numericas llegaron como texto. Se convierten con `errors="coerce"`,
que transforma en nulo cualquier valor que no sea convertible, en lugar de interrumpir la
ejecucion. Eso deja los problemas visibles como nulos en vez de esconderlos.

Ademas se reemplaza la cadena literal `"NaN"` por un nulo real. Sin este paso, `"NaN"` seria
tratada como una categoria valida mas del clima y del trafico, y terminaria convertida en una
columna binaria por el codificador, contaminando el modelo con una categoria que en realidad
significa "no sabemos".

In [ ]:
df = df.rename(columns={
    "Road_traffic_density": "trafico",
    "Delivery_person_Age": "edad",
    "Delivery_person_Ratings": "calificacion",
    "Vehicle_condition": "estado_vehiculo",
    "multiple_deliveries": "pedidos_simultaneos",
    "Type_of_vehicle": "vehiculo",
    "Type_of_order": "tipo_pedido",
    "City": "ciudad",
    "Festival": "festivo",
})

df = df.replace(["NaN", "nan", ""], np.nan)

for col in ["minutos", "edad", "calificacion", "pedidos_simultaneos", "estado_vehiculo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

nulos_reales = df[["minutos", "edad", "calificacion", "pedidos_simultaneos",
                   "trafico", "clima", "ciudad", "festivo"]].isna().sum()
print("Nulos reales tras destapar los disfrazados:")
print(nulos_reales[nulos_reales > 0].to_string())

### Etapa 3: construccion de la distancia

La distancia no viene en el archivo. Estan las coordenadas del restaurante y del destino, y
la distancia se deriva de ellas con la formula de Haversine, que calcula la separacion sobre
la superficie de una esfera en lugar de sobre un plano. Sobre las distancias de este conjunto,
por debajo de treinta kilometros, la diferencia frente a una aproximacion euclidiana sobre
grados es inferior al uno por ciento, y el costo computacional de ambas es equivalente.

Se toma el valor absoluto de cada coordenada. En el archivo original algunas latitudes y
longitudes vienen con el signo invertido respecto de otras del mismo par de ciudades, lo que
produciria distancias imposibles de miles de kilometros entre dos puntos de la misma urbe.

La distancia resultante es en linea recta, no por calle. Subestima el recorrido real en un
factor que depende del trazado urbano, pero como ese factor es aproximadamente constante
dentro de una misma ciudad, el modelo lineal lo absorbe en el coeficiente sin perder
capacidad predictiva.

In [ ]:
RADIO_TIERRA_KM = 6371.0


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * RADIO_TIERRA_KM * np.arcsin(np.sqrt(a))


df["distancia_km"] = haversine(
    df["Restaurant_latitude"].abs(), df["Restaurant_longitude"].abs(),
    df["Delivery_location_latitude"].abs(), df["Delivery_location_longitude"].abs(),
)

print(df["distancia_km"].describe().round(2).to_string())

### Etapa 4: tratamiento de los nulos

Se eliminan las filas con ausencias en la variable objetivo o en las variables numericas
predictoras. La decision se justifica por dos motivos: las ausencias son pocas, menos del 4 %
por columna, y estan repartidas sin patron aparente entre columnas distintas. Imputar la
media en una calificacion de repartidor o en la edad introduciria observaciones que nunca
existieron sobre una base amplia de datos completos, sin ganancia a cambio.

Para las categoricas se toma la decision opuesta: sus ausencias se imputan mas adelante,
dentro del `Pipeline`, con la categoria mas frecuente. La razon es que en produccion el
sistema tiene que responder aunque falte el dato de ciudad, y el `Pipeline` tiene que saber
que hacer en ese caso. Imputar afuera dejaria al modelo sin esa defensa.

In [ ]:
antes_nulos = len(df)
df = df.dropna(subset=["minutos", "edad", "calificacion", "pedidos_simultaneos",
                       "estado_vehiculo", "trafico", "clima"])
print(f"Filas eliminadas por nulos: {antes_nulos - len(df)}  ({100*(antes_nulos-len(df))/antes_nulos:.2f} %)")
print("Nulos que quedan en categoricas, para imputar dentro del Pipeline:")
print(df[["vehiculo", "tipo_pedido", "ciudad", "festivo"]].isna().sum().to_string())

### Etapa 5: valores atipicos e inconsistencias

**Primero, tres guardas de rango.** Se descartan distancias menores a medio kilometro o
mayores a treinta, y calificaciones fuera de la escala de uno a cinco. Ninguna de las tres
elimina filas en este conjunto, y reportarlo igual tiene sentido: deja constancia de que la
verificacion se hizo y de que el resultado fue limpio. Una guarda que no dispara sigue siendo
informacion.

In [ ]:
n0 = len(df)
df = df[df["distancia_km"] >= 0.5]
n1 = len(df)
df = df[df["distancia_km"] <= 30]
n2 = len(df)
df = df[(df["calificacion"] >= 1) & (df["calificacion"] <= 5)]
n3 = len(df)

print(f"Distancia menor a 0,5 km (coordenadas en cero): {n0 - n1:5d} filas")
print(f"Distancia mayor a 30 km (fuera del alcance):    {n1 - n2:5d} filas")
print(f"Calificacion fuera del rango 1 a 5:             {n2 - n3:5d} filas")

### Coordenadas anonimizadas: el problema que las guardas no detectan

Un grupo de filas trae las coordenadas del restaurante en cero exacto. No son un error de
captura sino una anonimizacion parcial: al restaurante se le borro la ubicacion y el destino
quedo guardando el desplazamiento respecto de ese origen. La distancia que sale de ahi es
aproximadamente correcta, porque el desplazamiento es lo que la formula usa, y por eso ninguna
guarda de rango las detecta: caen dentro de los limites igual que las filas normales.

Pero hay un sesgo. La formula de Haversine pondera la diferencia de longitud por el coseno de
la latitud, que corrige el hecho de que los meridianos se juntan hacia los polos. Con la
latitud en cero ese coseno vale uno, mientras que en las latitudes reales del conjunto, entre
doce y treinta grados, vale entre 0,98 y 0,87. La distancia de esas filas queda sobreestimada
entre un 2 y un 15 por ciento, y esa sobreestimacion es ruido inyectado en la variable que la
hipotesis pone a prueba.

La decision no se toma por principio sino midiendo. Se entrena el modelo de regresion lineal
multiple con y sin ellas, y se compara sobre el conjunto de prueba.

In [ ]:
df["coord_anonimizada"] = df["Restaurant_latitude"].abs() < 1

print(f"Filas con coordenadas anonimizadas: {int(df['coord_anonimizada'].sum())} "
      f"({100*df['coord_anonimizada'].mean():.1f} % del conjunto)")
print("\nComparacion con el resto:")
print(df.groupby("coord_anonimizada")[["distancia_km", "minutos"]].median().round(2).to_string())

for lat in [12, 22, 30]:
    sesgo = 100 * (1 / np.cos(np.radians(lat)) - 1)
    print(f"  A {lat} grados de latitud, la distancia queda sobreestimada un {sesgo:.1f} %")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

NUMERICAS = ["distancia_km", "edad", "calificacion", "pedidos_simultaneos", "estado_vehiculo"]
CATEGORICAS = ["trafico", "clima", "vehiculo", "tipo_pedido", "ciudad", "festivo"]


def ensayo(datos, etiqueta):
    Xtr, Xte, ytr, yte = train_test_split(
        datos[NUMERICAS + CATEGORICAS], datos["minutos"],
        test_size=0.2, random_state=RANDOM_STATE)
    prep = ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                          ("s", StandardScaler())]), NUMERICAS),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                          ("o", OneHotEncoder(drop="first", handle_unknown="ignore"))]), CATEGORICAS),
    ])
    m = Pipeline([("prep", prep), ("reg", LinearRegression())]).fit(Xtr, ytr)
    pred = m.predict(Xte)
    print(f"  {etiqueta:26s} n={len(datos):6d}   R2 {r2_score(yte, pred):.4f}   "
          f"RMSE {np.sqrt(mean_squared_error(yte, pred)):.2f} min")


print("Desempeno del modelo lineal multiple segun la decision:")
ensayo(df, "conservandolas")
ensayo(df[~df["coord_anonimizada"]], "eliminandolas")

**Decision: se eliminan.** Quitarlas mejora el R2 sobre el conjunto de prueba y reduce el
error tipico, pese a costar un ocho por ciento de las observaciones. Con mas de treinta y
nueve mil filas restantes el volumen sigue siendo amplio, de modo que la perdida no compromete
nada y la ganancia en calidad de la variable principal si se nota.

Vale registrar que la decision se tomo despues de medirla y no antes. Si el resultado hubiera
sido el contrario, conservarlas habria sido lo correcto: un ocho por ciento de datos reales
pesa mas que una imperfeccion que el modelo tolera.

In [ ]:
antes_anon = len(df)
df = df[~df["coord_anonimizada"]].drop(columns=["coord_anonimizada"])
print(f"Filas eliminadas por coordenadas anonimizadas: {antes_anon - len(df)}")

### Resumen de la limpieza

La codificacion de las variables categoricas no se hace aca. Va dentro del `Pipeline` del
notebook 04, junto con el escalado, para que ambas transformaciones se ajusten unicamente con
el conjunto de entrenamiento. Codificar ahora, sobre el conjunto completo, seria una fuga de
datos: las categorias presentes en el test estarian influyendo en como se representa el train.

In [ ]:
df = df[NUMERICAS + CATEGORICAS + ["minutos"]].reset_index(drop=True)

print(f"Filas: {filas_iniciales}  ->  {len(df)}   ({100*len(df)/filas_iniciales:.1f} % conservado)")
print(f"Columnas finales: {len(df.columns)}  ({len(NUMERICAS)} numericas, "
      f"{len(CATEGORICAS)} categoricas, 1 objetivo)")
df.head()

## 3.3 Division train/test

**Proporcion elegida: 80 % entrenamiento y 20 % prueba.** La justificacion es el volumen. Un
20 % sobre mas de cuarenta mil filas deja mas de ocho mil observaciones de prueba, cantidad
mas que suficiente para que las metricas sean estables: el error estandar de un promedio
sobre ocho mil casos es despreciable frente a las diferencias entre modelos que se van a
comparar. Reservar mas seria quitarle datos al entrenamiento sin ganar precision en la
medicion.

**Semilla fija en 42.** Garantiza que la particion sea identica en cada corrida y que
cualquiera pueda reproducir exactamente estos resultados.

**Sobre la estratificacion.** Corresponde preguntarse si hace falta. En clasificacion
la estratificacion mantiene la proporcion de clases; en regresion no hay clases que mantener.
Aun asi, la variable objetivo tiene una distribucion asimetrica, y una particion aleatoria
podria dejar los tiempos extremos mal repartidos. Por eso se estratifica sobre los cuartiles
del tiempo de entrega: se agrupa `minutos` en cuatro tramos y se pide que cada tramo mantenga
su proporcion en ambos conjuntos. Se obtiene la garantia de la estratificacion sin inventar
clases que el problema no tiene.

In [ ]:
from sklearn.model_selection import train_test_split

tramos = pd.qcut(df["minutos"], q=4, labels=False, duplicates="drop")

X = df[NUMERICAS + CATEGORICAS]
y = df["minutos"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=tramos)

print("Entrenamiento:", X_train.shape[0], "filas")
print("Prueba:       ", X_test.shape[0], "filas")

### Verificacion de la particion

Si la estratificacion funciono, la distribucion del tiempo de entrega debe ser practicamente
identica en ambos conjuntos. Comparar las medias no alcanza, porque dos distribuciones
distintas pueden compartir la media; por eso se comparan tambien los cuartiles.

In [ ]:
comparacion = pd.DataFrame({
    "train": y_train.describe(),
    "test": y_test.describe(),
}).round(3)
comparacion["diferencia"] = (comparacion["train"] - comparacion["test"]).round(3)
comparacion

**Lectura.** Media, desviacion y los tres cuartiles coinciden hasta la primera o segunda
cifra decimal entre ambos conjuntos. La particion quedo balanceada y cualquier diferencia de
desempeno entre train y test que aparezca despues no se podra atribuir a un sorteo desafortunado.

In [ ]:
X_train.to_csv(os.path.join(RUTA, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(RUTA, "X_test.csv"), index=False)
y_train.to_frame("minutos").to_csv(os.path.join(RUTA, "y_train.csv"), index=False)
y_test.to_frame("minutos").to_csv(os.path.join(RUTA, "y_test.csv"), index=False)

print("Particiones guardadas en:", RUTA)
for archivo in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    ruta = os.path.join(RUTA, archivo)
    print(f"  {archivo:14s} {os.path.getsize(ruta)/1024:8.1f} KB")